In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)

%pwd



In [ ]:
import pandas as pd
import seaborn as sns 
from scipy.stats import linregress
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import truncnorm

import traffic.utils.distribution_utils as du 

# time preference

In [ ]:
# lets call 7 am step 0 
# thus 10 am is
step_ten_am = (10-7)*3600
step_ten_am 
time_samp = du.make_truncnorm(10800, 0, 3000).rvs(1000)

sns.histplot(time_samp)

# analysis of cars IE learning the perams 

In [ ]:
# acceleration vs velosity curve generally looks flat followed by an exponential decay function
# acceleration is capped at a max value, mostly dependent on awd vs 2wd
# more hp and less weight has similar effects -> increasing the %max speed at witch the exponential decay of acceleration occures

# road angle affects the value of the max acceleration, the steeper the angle, the lower the max accel

# toyota 170hp, 2800lbs, max speed: 140, eol:20mph
# subaru outback, 200hp 3700lbs, max speed: 144, eol: 15mph
# BMW M3, 380hp, 3900lbs, max 170, eol 30mph
# nice corvet 490hp, 3600, max speed: 198, eol:45mph
# bugati 1500hp, 4400, max speed: 280, eol:108mph



    # this comes from 1.2*the average acceleration from a 0-60 time, 1.2 because max acceleration is reported for high end cars and i applied this to the rest

sample_cars = {
    'car': ['crv', 'tacoma', 'f-150', 'yugo', 'corolla', 'outback', 'BMW M3', 'Corvette C8'],
    "hp":  [190,    159,      290,     55,     170,       200,       380,      490],
    "lbs": [3500,   4200,     4500,    1800,   2800,      3700,      3900,     3600],
"max_speed":[130,   130,      159,     90,     140,       144,       170,      198], 
    "eol": [15,     10,       21,      17,     20,        15,        30,       45], 
'zero2sixty':[8.1,  7.1,      6.0,     14.0,   9.3,       6.0,       3.9,      2.8],
'max_accel': [ 4.3, 4.56, 6, 2.3, 3.9, 5.4, 8, 9] # kinda fake 
}

# yugo                               ~1.7 – 1.9     2.3
# Toyota Corolla	8.3 – 9.0	     ~3.0 – 3.3     3.9
# Subaru Outback XT	5.9 – 6.3	     ~4.2 – 4.5     5.4
# Honda CR-V	7.4 – 8.1	         ~3.3 – 3.6     4.3
# Toyota Tacoma	7.0 – 7.7	         ~3.5 – 3.8     4.56
# Ford F-150	5.3 – 6.0	         ~4.5 – 5.0     6
# BMW M3	2.7 – 3.9	             ~6.9 – 8.2     8
# Chevrolet Corvette C8	2.8	         ~7.6           9
# Bugatti Chiron	2.3 – 2.4	     ~9.9 – 10.2    12.

sample_cars_df = pd.DataFrame(sample_cars)
sample_cars_df = sample_cars_df[sample_cars_df.car != 'Bugatti'] # this turned out to really skew the dist

# calculate some composite cols
sample_cars_df['hp_per_lb'] = sample_cars_df.hp/sample_cars_df.lbs
sample_cars_df['eol_per_max_speed'] = sample_cars_df.eol/sample_cars_df.max_speed

# this is the proformance index, it's indexed from the yugo to the Corvette C8
highest_hp_per_lb =sample_cars_df.hp_per_lb.max()
lowest_hp_per_lb =sample_cars_df.hp_per_lb.min()

sample_cars_df['performance'] = 100*(sample_cars_df.hp_per_lb-lowest_hp_per_lb)/(highest_hp_per_lb-lowest_hp_per_lb)


sample_cars_df


In [ ]:
def estimate_max_accel(zero_to_sixty_time):
    return (26.82 / zero_to_sixty_time)   # m/s²
    

sample_cars_df['est_max_accel'] = sample_cars_df.zero2sixty.apply(estimate_max_accel)
sample_cars_df

In [ ]:
# Choose your y variables
y_vars = ['eol', 'max_speed', 'zero2sixty', 'est_max_accel']
x_var = 'performance'

# Create subplots
fig, axes = plt.subplots(1, 4, figsize=(15, 5), sharex=True)

# Generate each regplot
for ax, y in zip(axes, y_vars):
    sns.regplot(data=sample_cars_df, x=x_var, y=y, ax=ax)
    ax.set_title(f'{y} vs {x_var}')

plt.tight_layout()
plt.show()

In [ ]:
# essentially these are my learned coefficents for the perameters of the acceleration curve
slope_eol, intercept_eol, r_value_eol, p_value_eol, std_err_eol = linregress(sample_cars_df['performance'], sample_cars_df['eol'])
print(f"Best-fit equation: eol = {intercept_eol:.2f} + {slope_eol:.2f} × proformance")


slope_max_speed, intercept_max_speed, r_value_max_speed, p_value_max_speed, std_err_max_speed = linregress(sample_cars_df['performance'], sample_cars_df['max_speed'])
print(f"Best-fit equation: max_speed = {intercept_max_speed:.2f} + {slope_max_speed:.2f} × performance")

slope_z2s, intercept_z2s, r_value_z2s, p_value_z2s, std_err_z2s = linregress(sample_cars_df['performance'], sample_cars_df['zero2sixty'])
print(f"Best-fit equation: zero2sixty = {intercept_z2s:.2f} + {slope_z2s:.2f} × performance")

slope_max_accel, intercept_max_accel, r_value_max_accel, p_value_mmax_accel, std_err_max_accel = linregress(sample_cars_df['performance'], sample_cars_df['est_max_accel'])
print(f"Best-fit equation: est_max_accel = {intercept_max_accel:.2f} + {slope_max_accel:.2f} × performance")




In [ ]:
def performance_stats(performance):
    """these perameter values are coppied and pasted from the car performance analysis"""
    max_speed = 119.5 + 0.83 * performance  # mph
    end_linear = 10.35 + 0.33 * performance # mph
    zero2sixty = 10.00 + -0.08 * performance # secounds
    max_accel = 2.17 + 0.07 * performance #m/s^2 

    
    return max_speed, end_linear, zero2sixty, max_accel



# acceleration curve functions

In [ ]:
# Exp decay
def build_accel_function(performance, epsilon=0.01):
    max_speed, end_linear, zero2sixty, max_accel = performance_stats(performance)

    decay_range = max_speed - end_linear
    k = -np.log(epsilon / max_accel) / decay_range

    def accel(speed_mph):
        if speed_mph <= end_linear:
            return max_accel
        else:
            return max_accel * np.exp(-k * (speed_mph - end_linear))

    return accel


In [ ]:
import numpy as np
from scipy.optimize import minimize_scalar

# peace wise with 0-60 and max speed targeting
def build_piecewise_accel_function(performance):
    """
    Constructs a piecewise acceleration function:
    - Flat: [0, end_linear] -> constant max_accel
    - Decline to a60: [end_linear, 60]
    - Decline to 0: [60, max_speed]

    Parameters:
    - max_speed (float): Maximum speed (mph)
    - end_linear (float): End of flat acceleration region (mph)
    - zero2sixty (float): Time to reach 60 mph (seconds)
    - max_accel (float): Maximum acceleration (m/s^2)

    Returns:
    - accel_func (function): A function a(v) giving acceleration at speed v (mph)
    """
    max_speed, end_linear, zero2sixty, max_accel = performance_stats(performance)


    
    def time_error(a60_guess):
        # Part 2 coefficients
        m2 = (a60_guess - max_accel) / (60 - end_linear)
        b2 = max_accel - m2 * end_linear

        # Time from 0 to end_linear at constant acceleration
        t1 = end_linear / max_accel

        # Time from end_linear to 60 using ∫dv/a(v)
        v = np.linspace(end_linear, 60, 500)
        a_vals = m2 * v + b2
        t2 = np.trapz(1 / a_vals, v)

        return abs((t1 + t2) - zero2sixty)

    # Optimize to find best a60
    result = minimize_scalar(time_error, bounds=(0.1, max_accel), method='bounded')
    a60 = result.x

    # Part 2 coefficients
    m2 = (a60 - max_accel) / (60 - end_linear)
    b2 = max_accel - m2 * end_linear

    # Part 3 coefficients
    m3 = -a60 / (max_speed - 60)
    b3 = a60 - m3 * 60


    #print(f'A1: {max_accel}, A2={round(m2,2)}v+{round(b2,2)},   A3={round(m3,2)}v+{round(b3,2)} ')
    
    def accel(v):
        if v <= end_linear:
            return max_accel
        elif v <= 60:
            return m2 * v + b2
        elif v <= max_speed:
            return m3 * v + b3
        else:
            return 0.0

    return accel





In [ ]:
def build_parametric_accel_function(perf, c=20, n=2):
    """
    Parametric decay acceleration function:
        a(s) = a_max / (1 + (c / (s - s0))**n)

    Parameters:
    - a_max: peak acceleration at high speed (float)
    - c: shape constant (float)
    - s0: speed offset to shift the curve right (float)
    - n: decay steepness (int or float)

    Returns:
    - function taking speed s (mph) and returning acceleration (m/s^2)
    """

    max_speed, end_linear, zero2sixty, max_accel = performance_stats(perf)


    
    def accel(s):
        if s <= end_linear:
            return max_accel
        else:
            return  max_accel / (1 + ((s - end_linear) / c) ** n)

    return accel


In [ ]:
from scipy.stats import truncnorm
from scipy.stats import skewnorm

pctile = .2 
n = 10000

dist0 = skewnorm(loc=1, scale=0.35, a=1)
sample0=dist0.rvs(n)
deltav0 = sample0* 1 * 2.237


dist1 = skewnorm(loc=2.5, scale=0.4, a=1)
sample1=dist1.rvs(n,random_state=1)
deltav1 = sample1 * 2.237*2

dist2 = skewnorm(loc=2, scale=0.4, a=1)
sample2=dist2.rvs(n)
deltav2 = sample2 * 2.237 * 2

dist3 = skewnorm(loc=1.5, scale=0.3, a=1)
sample3=dist3.rvs(n)
deltav3 = sample3 * 2.237 * 2


mph0 = deltav0
mph1 = deltav0 + deltav1
mph2 = deltav0 + deltav1 + deltav2
mph3 = deltav0 + deltav1 + deltav2 + deltav3


# mean = .12
# #sample = truncnorm((1.2 - 1.5)/.2, (2.5 - 1.5)/.2, loc=1.5, scale=.2).rvs(100)


stage = mph3
sns.histplot(mph3)

mean = round(np.mean(stage),2)
variance = round(np.var(stage),2)


print(f'mean:{mean, variance} +- 1sd:[{mean-variance}, {mean+variance}]] ')



dist0.ppf(2)

In [ ]:
from scipy.stats import truncnorm
from scipy.stats import skewnorm

pctile = .2 
n = 10000




l = [0.35, .4, .4, .3]
[i*1.5 for i in l ]

In [ ]:
# approvimating this study https://www.jsheld.com/insights/articles/a-naturalistic-study-of-vehicle-acceleration-and-deceleration-at-an-intersection
def build_empirical_accel_function(pctile, mean_shift=-.25 , var_streach=1.5):
    """
    Builds a function that estimates acceleration (in m/s²)
    given speed (in mph), using empirical acceleration data
    from real-world stop sign behavior.

    Returns:
        accel(speed_mph): callable function
    """
    trimmed_pctile = np.clip(pctile, .07, .95)
    og_means = [1, 2.5, 2, 1.5]
    og_vars = [.35, .4, .4, .3]

    means = [i+mean_shift for i in og_means]
    var = [i*var_streach for i in og_vars]
    
    # differnet dists in m/s^s 
    dist0 = skewnorm(loc=means[0], scale=var[0], a=1)
    dist1 = skewnorm(loc=means[1], scale=var[1], a=1)
    dist2 = skewnorm(loc=means[2], scale=var[2], a=1)
    dist3 = skewnorm(loc=means[3], scale=var[3], a=1)

    #  # differnet dists in m/s^s 
    # dist0 = skewnorm(loc=1, scale=0.52, a=1)
    # dist1 = skewnorm(loc=2.5, scale=0.6, a=1)
    # dist2 = skewnorm(loc=2, scale=0.6, a=1)
    # dist3 = skewnorm(loc=1.5, scale=0.45, a=1)
    
    # Acceleration values in G, time intervals in seconds
    segments = [
        {"start_t": 0, "end_t": 2, "accel_mpss": dist0.ppf(trimmed_pctile)},
        {"start_t": 2, "end_t": 4, "accel_mpss": dist1.ppf(trimmed_pctile)},
        {"start_t": 4, "end_t": 6, "accel_mpss": dist2.ppf(trimmed_pctile)},
        {"start_t": 6, "end_t": 8, "accel_mpss": dist3.ppf(trimmed_pctile)},
    ]

    # Convert to speed ranges in mph
    speed_bounds = [0]
    for seg in segments:
        delta_v_mps = seg["accel_mpss"] * (seg["end_t"] - seg["start_t"])
        delta_v_mph = delta_v_mps * 2.2  # convert m/s to mph
        speed_bounds.append(speed_bounds[-1] + delta_v_mph)

    # Pre-compute acceleration in m/s² for each segment
    accels_mps2 = [seg["accel_mpss"] for seg in segments]
    def accel(speed_mph):
        for i in range(len(speed_bounds) - 1):
            if speed_bounds[i] <= speed_mph < speed_bounds[i + 1]:
                return accels_mps2[i]
        return np.clip(accels_mps2[-1]*  (1-((speed_mph-speed_bounds[-1])/70)), 0,10)

    return accel



# Test accel functions

## Functions for testing 

In [ ]:
def plot_accel_curve(perf, func):
    accel_curve = func(perf) 

    speeds = np.linspace(0, 100, 200)
    accel_points = [accel_curve(s) for s in speeds]

    plt.plot(speeds, accel_points)
    # Set y-axis range from 0 to 10
    plt.show()


def plot_multi_accel_curve(accel_func_name, performance_levels = range(0, 50, 5)):

    plt.figure(figsize=(10, 6))
    
    for perf_val in performance_levels:
        # make the unique function
        accel_fn = accel_func_name(perf_val)
        # define the other stuff for plotting
        max_speed = 119.48 + 0.83*perf_val
        speeds = np.linspace(0, 200 + 10, 200)
        accels = [accel_fn(s) for s in speeds]
        plt.plot(speeds, accels, label=f'Perf {perf_val}')
    
    plt.xlabel("Speed (mph)")
    plt.ylabel("Acceleration (m/s²)")
    plt.title("Acceleration vs Speed for Different Performance Levels")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

def simulate_speed_plot(perf, accel_func_name,  dt=0.01, max_time=60):
    speed = 0
    speeds = [0]
    times = [0]
    speed_at_0_2_60_time=None 
    time2sixty=None
    
    accel_fn = accel_func_name(perf)
    
    max_speed, end_linear, zero2sixty, max_accel = performance_stats(perf)

    for t in np.arange(0, max_time, dt):
        a = accel_fn(speed)
        speed += a * dt * 2.237  # convert m/s² * s → mph
        speed = min(speed, max_speed)
        # save the time to 60
        if round(speed) == 60:
            time2sixty = round(t, 1)
            
        speeds.append(speed)
        times.append(t + dt)

        # save the speed at reported zero2sixty time
        if round(t,2) == round(zero2sixty,2):
            speed_at_0_2_60_time = round(speed,1)
        
        if speed >= max_speed - 0.01:
            break

    if speed_at_0_2_60_time:
        print(f'Speed at zero2sixty time:{speed_at_0_2_60_time}mph')
    if time2sixty:
        print(f'Time till 60mph:{time2sixty}s)')

    plt.plot(times, speeds)
    plt.xlabel("Time (s)")
    plt.ylabel("Speed (mph)")
    plt.title("Simulated Speed Over Time")
    plt.grid(True)
    plt.show()

## Define curve and perf to test

In [ ]:

# build_parametric_accel_function
# build_piecewise_accel_function
# build_accel_function

#build_empirical_accel_function

perf = .3
max_speed, end_linear, zero2sixty, max_accel = performance_stats(perf)
print(f'max_speed:{max_speed}, end_linear:{end_linear}, zero2sixty:{zero2sixty}, max_accel:{max_accel}')

accel_func_name = build_empirical_accel_function


plot_accel_curve(perf, accel_func_name)
simulate_speed_plot(perf, accel_func_name, max_time=30)


plot_multi_accel_curve(accel_func_name, [.1, .2, .3, .4, .5, .6, .8, 1])


In [ ]:
[.1, .2, .3, .4, .5, .6]


# Breaking

In [ ]:
# gap is smaller than ideal gap 
def get_deceleration(how='soft'):
    options = {'soft': hf.get_mps(1), #~1mph/s 
               'normal': hf.get_mps(2.5), #~2.5mph/s
               'hard': hf.get_mps(5) #~5mph/s
              }
    return  options[how]


def less_smooth_brake( gap, ideal_gap):
        """
        Simulate more realistic, human-like braking behavior.
        Returns a value between 0 and 1 indicating brake intensity.
        """
        if ideal_gap <= 0 or np.isnan(ideal_gap):
            return 0
        force = max((ideal_gap - gap) / ideal_gap, 0)
        # Squared to overreact when too close
        base = force ** 2
        
        # Add some human-like noise
        noise = np.random.normal(0, .1)
        break_pct =  np.clip(base + noise, 0,1)

        deceleration = break_pct * 4 # <- this is acting as max decel 
        return deceleration

def speed_limit_brake( speed_limit, speed):
        """
        Simulate more realistic, human-like braking behavior.
        Returns a value between 0 and 1 indicating brake intensity.
        """
        
        force = (speed-speed_limit)/speed_limit
        # Squared to overreact when too close
        base = force ** 2
        
        # Add some human-like noise
        noise = np.random.normal(0, .1)
        break_pct =  np.clip(base + noise, 0,1)

        deceleration = break_pct * 4 # <- this is acting as max decel 
        return deceleration

In [ ]:
def speed_limit_brake(speed_limit, speed):
    if speed < speed_limit:
        # this should never be triggered but i added anyway to make sure it didnt trip an error
        return 0
    mph_over = hf.get_mph(speed)-hf.get_mph(speed_limit) # used pct over because speed_limit and speed come in in mps 
    #print(mph_over)
    if mph_over > 7: 
        return 1.1
    elif mph_over > 2:
        return .5
    elif mph_over > 0:
        return .2
        

# sl_range = range(20, 60,5)
# speed_range = range(30, 60, 2)

# hf.plot_param_grid_heatmap(sl_range, speed_range, speed_limit_brake_test, 'speed_limit', 'speed', fixed_params=None, round_to=2)

9.81*.19
# hf.get_mps(2.5)
# 9.81*.05


time_range = range(0, 60,1)
speed = hf.get_mps(60)
speed_list = []

for i in time_range:
    #print(speed_limit_brake_test(speed_limit=hf.get_mps(45), speed=speed))
    speed-=speed_limit_brake_test(speed_limit=hf.get_mps(45), speed=speed)
    speed_list.append(speed)

sns.scatterplot(x=time_range, y=speed_list)

hf,get

In [ ]:
from scipy.stats import uniform

ideal_gap = 100
gap = uniform(0, 100).rvs(1000)

deceleration = [less_smooth_brake(gap=i, ideal_gap=ideal_gap) for i in gap] 

sns.scatterplot(x=gap, y=deceleration)
plt.xlabel("Gap (m)")
plt.ylabel("Deceleration (m/s^2)")
plt.title("Possible Decelerations by gap")
plt.grid(True)

# ideal follow distance

In [ ]:
fs = truncnorm((1.2 - 1.5)/.2, (2.5 - 1.5)/.2, loc=1.5, scale=.2).rvs(1000)# - .75 

mph = uniform(0, 60).rvs(1000)
mps = [hf.get_mps(i) for i in mph] 
fd = mps * fs

fd_df = pd.DataFrame({'mph':mph, 'mps':mps,'fd':fd,})




fd_df['cat'] = (fd_df.mph//5)*5

fd_df

# curve adjust

In [ ]:

upper=1
lower=.65
mean= (upper+lower)/2
var = .1
def make_truncnorm(upper, lower, var, mean=None):
    '''
    If a mean is not passed the average of upper and lower will be used
    '''
    if not mean:
        mean = (upper+lower)/2

    return truncnorm((lower - mean)/var, (upper - mean)/var, loc=mean, scale=var)


a = make_truncnorm(upper,lower,var).rvs(10000)

sns.histplot(a)

In [ ]:
k_range = np.linspace(0.45, .7, 4)
curve_range = range(0,90,10)
speed_range = range(0,70,10)


hf.plot_param_grid_heatmap(var1_vals=speed_range, var2_vals=curve_range, func=curve_adjust, param1_name='speed', param2_name='curvature', fixed_params={'max_affect_pct':.9})